# Explainable Android Malware Detection - Neural Network Baseline

This notebook covers the training and evaluation of the baseline Neural Network (Multilayer Perceptron) on the `cleaned_dataset.csv`.
Deep learning models require feature scaling and are structured to contrast with the tree-based models (Random Forest, XGBoost).

In [ ]:
# Step 1: Import necessary libraries
import pandas as pd
import numpy as np
import time
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure tensorflow is installed: pip install tensorflow
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
except ImportError:
    print("TensorFlow is not installed. You can install it using: pip install tensorflow")

# Create directories for saving models and data mappings
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)

# Optional: SHAP for explainability (ensure shap is installed: pip install shap)
try:
    import shap
except ImportError:
    print("SHAP is not installed. You can install it using: pip install shap")

### Load the Cleaned Dataset

In [ ]:
# Step 2: Load the dataset
data_path = 'cleaned_dataset.csv'
print(f"Loading data from {data_path}...")
df = pd.read_csv(data_path)

print(f"Dataset shape: {df.shape}")
print(f"Class distribution:\n{df['Label'].value_counts(normalize=True) * 100}")
df.head()

### Data Preprocessing & Splitting
Unlike Tree-based models, Neural Networks require feature scaling so that features with large frequency values don't dominate the weight updates.

In [ ]:
# Step 3: Separate features (X), target (y), and IDs
id_col = 'apk_id'
ids = df[id_col]
X = df.drop(['Label', id_col], axis=1)
y = df['Label']

# Step 4: Train-Test Split (80% training, 20% testing)
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, ids, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

print(f"\nClass distribution in training set (Imbalance Check):\n{y_train.value_counts()}")

# Save the test set ID mapping for later obfuscation matching
test_mapping_path = 'data/nn_test_set_ids.csv'
pd.DataFrame({'apk_id': id_test, 'label': y_test}).to_csv(test_mapping_path, index=False)
print(f"\nTest set IDs saved to {test_mapping_path}\n")

# Step 5: Feature Scaling and Persistence
# It's crucial to fit the scaler ONLY on the training data to prevent data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

scaler_path = 'models/nn_scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f"Scaler saved to {scaler_path}")

### Model Architecture, Training and Persistence
We build a Multilayer Perceptron (MLP) with a funnel architecture (128 -> 64 -> 32) and use Dropout to prevent overfitting. After training, we persist it to disk. We also compute and apply class weights to penalize misclassification of the minority class.

In [ ]:
# Step 6: Define the Neural Network Architecture
# Ensure consistent weight initialization using a seed
tf.random.set_seed(42)

nn_model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid') # Binary classification output
])

nn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

nn_model.summary()

In [ ]:
# Step 7: Train the Model with Early Stopping and Class Weights
# Compute class weights to handle imbalance
weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(weights))
print(f"Calculated class weights: {class_weights}\n")

early_stopping = EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    restore_best_weights=True
)

print("Training Neural Network...")
start_time = time.time()
history = nn_model.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.2,
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=1
)
training_time = time.time() - start_time
print(f"Training completed in {training_time:.2f} seconds.\n")

# Step 8: Save the trained model to disk
model_path = 'models/nn_baseline.keras'
nn_model.save(model_path)
print(f"Model saved to {model_path}")

# Evaluate model size
size_kb = os.path.getsize(model_path) / 1024
print(f"Model size: {size_kb:.2f} KB")

### Training History Visualization

In [ ]:
# Step 9: Plot Training & Validation Loss
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Neural Network Loss over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

### Performance Evaluation
Now we evaluate the model on the scaled, unseen test set.

In [ ]:
# Step 10: Make predictions and evaluate
print("Evaluating model on test set...")
start_time = time.time()
y_pred_prob = nn_model.predict(X_test_scaled)
inference_time = time.time() - start_time

# Convert probabilities to binary predictions (threshold = 0.5)
y_pred = (y_pred_prob > 0.5).astype(int)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_prob)

print("\n--- Model Performance ---")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"Inference time for {len(y_test)} samples: {inference_time:.4f} seconds")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Benign (0)', 'Malware (1)']))

### Confusion Matrix Visualization

In [ ]:
# Step 11: Plot the Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Benign', 'Malware'], yticklabels=['Benign', 'Malware'])
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.title('Neural Network Confusion Matrix')
plt.show()